# E4 · QLoRA (receta RAFT) para StructAI — issue #179

**Estado: sin ejecutar todavía.** Escrito con las funciones que documenta Unsloth (`FastLanguageModel`, `save_pretrained_gguf`); la API de Unsloth/TRL cambia seguido, así que la primera corrida valida cada celda.

- **Objetivo:** que el modelo use el contexto de StructAI, cite y se abstenga. **No** memorizar normas (Ovadia 2024, Gekhman 2024).
- **Uso:** solo CI y evaluaciones; el modelo **no va a producción**.
- **Datos:** `scripts/finetuning/datos/` (normas públicas + respuestas verificadas; sin datos de usuarios).
- **Entorno:** Kaggle (GPU T4 ×2 o P100, ~30 h/semana) para corridas; Colab (T4) para iterar. Activar la GPU antes de correr.
- **Criterio preregistrado:** se adopta solo si mejora los hechos correctos con IC que no cruce 0 **y no aumenta** la alucinación numérica, medido en el homelab con `correr_bateria.py`.

In [ ]:
# 1) Dependencias (Colab/Kaggle con GPU)
!pip install -q unsloth
import torch; print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# 2) Datos desde el repo (rama del benchmark)
!git clone -q --depth 1 -b eval/golden-sabotaje https://github.com/wilmerjoseperezorozco-dev/structai.git
DATOS = "structai/scripts/finetuning/datos"
import json
def leer(n): return [json.loads(l) for l in open(f"{DATOS}/{n}", encoding="utf-8")]
train, heldout, abst = leer("raft_train.jsonl"), leer("raft_heldout.jsonl"), leer("raft_heldout_abstencion.jsonl")
print(len(train), len(heldout), len(abst)); print(json.load(open(f"{DATOS}/MANIFIESTO.json"))["train"])

In [ ]:
# 3) Modelo base en 4 bits + adaptadores LoRA
from unsloth import FastLanguageModel
MODELO_BASE = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # alternativa más liviana: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
MAX_SEQ = 8192   # p95 del contexto ~17.500 caracteres (~5.000 tokens); si no cabe en memoria, bajar a 6144
model, tokenizer = FastLanguageModel.from_pretrained(MODELO_BASE, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=179)

In [ ]:
# 4) Plantilla de chat de Qwen; se excluyen los ejemplos más largos que MAX_SEQ (se reporta cuántos)
from datasets import Dataset
def a_texto(ej): return {"text": tokenizer.apply_chat_template(ej["messages"], tokenize=False)}
def preparar(filas):
    ds = Dataset.from_list([{"messages": f["messages"]} for f in filas]).map(a_texto)
    largos = [len(tokenizer(t).input_ids) for t in ds["text"]]
    keep = [i for i, n in enumerate(largos) if n <= MAX_SEQ]
    print(f"{len(filas) - len(keep)} de {len(filas)} exceden {MAX_SEQ} tokens y se excluyen")
    return ds.select(keep)
ds_train, ds_eval = preparar(train), preparar(heldout)

In [ ]:
# 5) Entrenamiento: pérdida solo sobre las respuestas, parada temprana con el held-out,
#    checkpoints a Drive (Colab) o /kaggle/working (Kaggle) por si la sesión se cae
import os
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
SALIDA = "/kaggle/working/e4" if os.path.exists("/kaggle") else "/content/drive/MyDrive/structai_e4"
if SALIDA.startswith("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
cfg = SFTConfig(
    output_dir=SALIDA, dataset_text_field="text", max_seq_length=MAX_SEQ,
    per_device_train_batch_size=1, gradient_accumulation_steps=8, num_train_epochs=2,
    learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
    eval_strategy="steps", eval_steps=10, save_strategy="steps", save_steps=10, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", logging_steps=5, seed=179,
    report_to="none")
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, eval_dataset=ds_eval, args=cfg,
                     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train(); print(stats)

In [ ]:
# 6) Chequeo rápido: ¿se abstiene cuando falta el dato? ¿responde cuando está?
FastLanguageModel.for_inference(model)
def generar(msgs, n=300):
    ids = tokenizer.apply_chat_template(msgs[:2], add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=n, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
abst_ok = sum("no encontr" in generar(f["messages"]).lower() for f in abst)
print(f"abstención correcta en el held-out sin oráculo: {abst_ok}/{len(abst)}")
for f in heldout[:3]:
    print("PREGUNTA:", f["messages"][1]["content"].rsplit("PREGUNTA:", 1)[-1][:150]); print("MODELO:", generar(f["messages"])[:400]); print("---")

In [ ]:
# 7) Exportar: adaptador LoRA (chico) + GGUF Q4_K_M para Ollama en el homelab
model.save_pretrained(f"{SALIDA}/lora"); tokenizer.save_pretrained(f"{SALIDA}/lora")
model.save_pretrained_gguf(f"{SALIDA}/gguf", tokenizer, quantization_method="q4_k_m")
!ls -la {SALIDA}/gguf

## 8) En el homelab (no en el notebook)

1. Descargar el `.gguf` (~4,7 GB para 7B) y el `Modelfile` que genera Unsloth a `C:\Users\wilme\modelos\structai_e4\` (desde WSL: `/mnt/c/Users/wilme/modelos/structai_e4/`).
2. En WSL: `ollama create structai-qwen7b-e4 -f Modelfile` y probar con `ollama run structai-qwen7b-e4 "hola"`.
3. Medir con el mismo harness y la misma batería que el modelo base:
   `python scripts/evaluacion/golden/e3_modelos_locales/correr_bateria.py --modelo structai-qwen7b-e4 --bateria todo`
4. Comparar contra `qwen2.5:7b` con McNemar pareado (hechos correctos) y revisar la alucinación numérica. Registrar el resultado en #179.